<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?**

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | 900 preguntas de Dolly con su respuesta reescrita en el estilo (listo) | ficha de datos en MLflow |
| Plantilla | la plantilla de chat y la máscara $m_t$ (**TODO 1**); la pérdida de la respuesta (**TODO 2**) | |
| Ajuste | LoRA a mano sobre $W_Q, W_K, W_V, W_O$, registrado en MLflow (listo) | `lora_estilo.pt` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**) | tabla en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 20 minutos) con **Internet** activado (el modelo viene de Hugging Face). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [ ]:
%pip install -q mlflow

In [ ]:
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, os, random, time, textwrap
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = dev == 'cuda'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· MLflow', mlflow.__version__, '·', dev)

## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las 2 200 del notebook 6.3. Del `train` separamos 100 para validar; `test` se usa solo en el TODO 3.

In [ ]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
val, train = train_all.iloc[:100].reset_index(drop=True), train_all.iloc[100:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

## 2. El modelo (listo)
`prompt_ids` aplica la plantilla de chat del modelo (con o sin mensaje de sistema) y termina en el turno del asistente; `responder` genera con greedy hasta `<|im_end|>`. Mira cómo responde el modelo con y sin el estilo en el sistema.

In [ ]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
try:
    modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32)
except TypeError:
    modelo = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float32)
modelo = modelo.to(dev).eval()
for p in modelo.parameters():
    p.requires_grad_(False)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def prompt_ids(pregunta, sistema=None):
    msgs = ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]
    texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(texto, add_special_tokens=False)['input_ids']

@torch.no_grad()
def responder(pregunta, sistema=None, max_nuevos=150):
    x = torch.tensor([prompt_ids(pregunta, sistema)], device=dev)
    out = modelo.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=max_nuevos,
                          do_sample=False, eos_token_id=FIN, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip()

print(tok.decode(prompt_ids('¿Qué es un átomo?')))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(q, ESTILO), 300))

## 3. TODO 1 · La plantilla y la máscara · 10 min
Escribe `codificar(pregunta, respuesta, sistema=None)`. Devuelve dos listas del mismo largo:
- `ids`: los ids del prompt (`prompt_ids(pregunta, sistema)`) seguidos de los de la respuesta y del token de fin `FIN` (`<|im_end|>`);
- `m`: 0 en cada token del prompt y 1 en cada token de la respuesta **y** en `FIN` (diapositiva 71: así el modelo aprende a parar).

Tokeniza la respuesta con `tok(respuesta.strip(), add_special_tokens=False)`.

In [ ]:
def codificar(pregunta, respuesta, sistema=None):
    # TODO 1: (ids, m) con la plantilla de chat; m_j = 1 solo en los tokens de la respuesta y en <|im_end|>
    raise NotImplementedError('Completa TODO 1')

In [ ]:
ids, m = codificar('¿Qué es un átomo?', 'Partícula mínima, vuestra merced.')
assert len(ids) == len(m) and set(m) <= {0, 1}, 'ids y m deben tener el mismo largo, y m solo 0 y 1.'
assert ids[:len(prompt_ids('¿Qué es un átomo?'))] == prompt_ids('¿Qué es un átomo?'), 'Los ids deben empezar con el prompt de prompt_ids.'
assert tok.decode([i for i, k in zip(ids, m) if k]) == 'Partícula mínima, vuestra merced.<|im_end|>', \
    'Los tokens con m = 1 deben ser exactamente la respuesta y <|im_end|>.'
assert m == sorted(m), 'La máscara debe ser 0 en el prompt y 1 de ahí en adelante.'
ids_s, m_s = codificar('¿Qué es un átomo?', 'Partícula mínima, vuestra merced.', ESTILO)
assert sum(m_s) == sum(m) and len(ids_s) > len(ids), 'Con sistema, el prompt crece y la respuesta no cambia.'
print('codificar correcta ·', sum(m), 'tokens de respuesta de', len(ids))

In [ ]:
T_MAX = 384

def lote(filas, sistema=None, columna='response_estilo'):
    cod = [codificar(r.instruction, getattr(r, columna), sistema) for r in filas.itertuples()]
    cod = [(i[:T_MAX], k[:T_MAX]) for i, k in cod]
    T = max(len(i) for i, _ in cod)
    ids = torch.full((len(cod), T), tok.eos_token_id, dtype=torch.long)
    m = torch.zeros((len(cod), T), dtype=torch.long); att = torch.zeros_like(m)
    for b, (i, k) in enumerate(cod):
        ids[b, :len(i)] = torch.tensor(i); m[b, :len(i)] = torch.tensor(k); att[b, :len(i)] = 1
    return ids.to(dev), m.to(dev), att.to(dev)

## 4. TODO 2 · La pérdida de la respuesta · 5 min
Escribe `perdida_respuesta(logits, ids, m)` a partir de las diapositivas 69 y 71. `logits` es $B \times T \times \lvert V\rvert$, `ids` y `m` son $B \times T$. La posición $t$ predice el token $t+1$:

$$L_t = -\log \mathrm{softmax}(\mathbf z_t)_{w_{t+1}}, \qquad J = \frac{\sum_t m_{t+1}\, L_t}{\sum_t m_{t+1}}$$

Usa `F.cross_entropy(..., reduction='none')` y calcula en `float`. La celda de prueba compara tu $J$ con la de la librería (`labels = -100` fuera de la respuesta).

In [ ]:
def perdida_respuesta(logits, ids, m):
    # TODO 2: J = sum_t m_{t+1} L_t / sum_t m_{t+1}, con L_t = -log softmax(z_t)[w_{t+1}]
    raise NotImplementedError('Completa TODO 2')

In [ ]:
z = torch.zeros(1, 4, 10)
assert abs(perdida_respuesta(z, torch.tensor([[1, 2, 3, 4]]), torch.tensor([[0, 0, 1, 1]])).item() - math.log(10)) < 1e-5, \
    'Con logits en cero, J debe ser ln|V|.'
z = torch.full((1, 3, 5), -10.0); z[0, 0, 2] = 10.0; z[0, 1, 4] = -10.0
J = perdida_respuesta(z, torch.tensor([[0, 2, 4]]), torch.tensor([[0, 1, 0]]))
assert J.item() < 1e-6, 'Solo cuenta la posición que predice el token 2, que el modelo acierta: J ≈ 0. ¿Corriste el objetivo un lugar?'
ids, m, att = lote(train.iloc[:4])
with torch.no_grad():
    logits = modelo(input_ids=ids, attention_mask=att).logits
    J = perdida_respuesta(logits, ids, m)
    J_hf = modelo(input_ids=ids, attention_mask=att, labels=ids.masked_fill(m == 0, -100)).loss
assert abs(J.item() - J_hf.item()) < 1e-3, f'Tu J ({J.item():.4f}) no coincide con la de la librería ({J_hf.item():.4f}).'
print(f'perdida_respuesta correcta · J = {J.item():.4f} · librería {J_hf.item():.4f}')
del logits

## 5. LoRA y entrenamiento (listo)
La clase `LoRA` del notebook 6.3, con $r = 16$ sobre las cuatro matrices de la atención. `entrenar` hace una época sobre las 900 respuestas con estilo y registra en MLflow los parámetros, $J$ de entrenamiento por paso y $J$ de validación cada 25 pasos; guarda `lora_estilo.pt` con las matrices $A_r$, $B$ y el **contrato**.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

In [ ]:
class LoRA(nn.Module):
    """Del notebook 6.3: y = W0 x + (alpha / r) B A_r x, con W0 congelada."""
    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.r, self.alpha, self.escala = base, r, alpha, alpha / r
        w = base.weight
        self.A = nn.Parameter(torch.randn(r, base.in_features, device=w.device) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=w.device))
        self.activa = True

    def forward(self, x):
        y = self.base(x)
        return y + self.escala * ((x @ self.A.T) @ self.B.T) if self.activa else y

OBJETIVOS = ('q_proj', 'k_proj', 'v_proj', 'o_proj')

def poner_lora(modelo, r=16, alpha=32):
    capas = {}
    for nombre, mod in list(modelo.named_modules()):
        for hijo in OBJETIVOS:
            lin = getattr(mod, hijo, None)
            if isinstance(lin, LoRA):                       # una segunda corrida parte otra vez de W0
                lin = lin.base
            if isinstance(lin, nn.Linear):
                capas[f'{nombre}.{hijo}'] = LoRA(lin, r, alpha)
                setattr(mod, hijo, capas[f'{nombre}.{hijo}'])
    return capas

@contextmanager
def sin_lora():
    for c in CAPAS.values(): c.activa = False
    try:
        yield
    finally:
        for c in CAPAS.values(): c.activa = True

@torch.no_grad()
def nll(datos, sistema=None, B=8):
    """El J de TODO 2 sobre todas las respuestas de `datos`, sin gradiente."""
    tot = n = 0.0
    for i in range(0, len(datos), B):
        ids, m, att = lote(datos.iloc[i:i + B], sistema)
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            logits = modelo(input_ids=ids, attention_mask=att).logits
        k = m[:, 1:].sum().item()
        tot += perdida_respuesta(logits, ids, m).item() * k; n += k
    return tot / n

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('S06-estilo-llm')
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}

def entrenar(nombre, r=16, alpha=32, lr=2e-4, epocas=1, B=8, cada=25):
    global CAPAS
    torch.manual_seed(0)
    CAPAS = poner_lora(modelo, r, alpha)
    params = [p for c in CAPAS.values() for p in (c.A, c.B)]
    n_ent = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    pasos = epocas * math.ceil(len(train) / B); warm = max(1, pasos // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, pasos - warm))))
    escalador = torch.amp.GradScaler('cuda', enabled=amp)
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.log_params({'modelo': MODELO, 'r': r, 'alpha': alpha, 'lr': lr, 'pasos': pasos, 'B': B,
                           'entrenables': n_ent, 'objetivos': ','.join(OBJETIVOS), **data_card})
        mlflow.log_metric('val_J', nll(val), step=0)
        modelo.train(); paso, t0 = 0, time.time()
        for ep in range(epocas):
            orden = np.random.default_rng(ep).permutation(len(train))
            for i in range(0, len(orden), B):
                ids, m, att = lote(train.iloc[orden[i:i + B]])
                with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                    J = perdida_respuesta(modelo(input_ids=ids, attention_mask=att).logits, ids, m)
                opt.zero_grad(set_to_none=True)
                escalador.scale(J).backward(); escalador.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                escalador.step(opt); escalador.update(); sched.step(); paso += 1
                mlflow.log_metric('train_J', J.item(), step=paso)
                if paso % cada == 0 or paso == pasos:
                    modelo.eval(); vj = nll(val); modelo.train()
                    mlflow.log_metric('val_J', vj, step=paso)
                    print(f'paso {paso:4d}/{pasos} · train {J.item():.3f} · val {vj:.3f} · {time.time() - t0:.0f} s')
        modelo.eval()
        ckpt = {'contrato': CONTRATO, 'r': r, 'alpha': alpha, 'objetivos': OBJETIVOS,
                'lora': {k: {'A': c.A.detach().cpu(), 'B': c.B.detach().cpu()} for k, c in CAPAS.items()}}
        torch.save(ckpt, 'lora_estilo.pt'); mlflow.log_artifact('lora_estilo.pt')
        mlflow.log_dict(CONTRATO, 'contrato.json')
        print(f'{n_ent:,} parámetros entrenables · {time.time() - t0:.0f} s')
        return run.info.run_id

In [ ]:
run_id = entrenar('1-lora-r16')

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(datos)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `datos` en tres condiciones (diapositiva 74):

| Clave | LoRA | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with sin_lora():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | encendido | ninguno |

`nll(datos, sistema)` ya calcula $J$ sobre todo un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [ ]:
def comparar(datos):
    # TODO 3: {'base': J, 'base + estilo en el prompt': J, 'LoRA': J} sobre las respuestas de `datos`
    raise NotImplementedError('Completa TODO 3')

In [ ]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with sin_lora():
    assert abs(r['base'] - nll(test.iloc[:8])) < 1e-4, '"base" debe medirse con LoRA apagado y sin sistema.'
assert abs(r['LoRA'] - nll(test.iloc[:8])) < 1e-4, '"LoRA" debe medirse con LoRA encendido y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt debe usar sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
    for _, f in tabla.iterrows():
        mlflow.log_metric('test_J_' + f['condición'].replace(' ', '_').replace('+', 'mas'), f['J_test'])
print(tabla.round(3).to_string(index=False))

In [ ]:
for q in test.instruction[:3]:
    print('▶', q)
    with sin_lora():
        print('  base:          ', textwrap.shorten(responder(q), 260))
        print('  base + estilo: ', textwrap.shorten(responder(q, ESTILO), 260))
    print('  LoRA:          ', textwrap.shorten(responder(q), 260), '\n')

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio y córrelo como `'2-...'` con `entrenar`; antes de correrlo, escribe tu predicción. Después repite `comparar(test)`.
- $r = 4$ o $r = 64$;
- LoRA solo en $W_Q$ y $W_V$ (cambia `OBJETIVOS`);
- la mitad de los datos de entrenamiento;
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo (`columna='response'` en `lote`): ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_

In [ ]:
# run_id_2 = entrenar('2-...', r=...)
# comparar(test)

## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, ckpt)`: recibe una pregunta en texto y el `ckpt` que guardó `entrenar` (descargado de MLflow), y devuelve la respuesta en texto. Usa solo lo que trae `ckpt`:
1. el **contrato**: la pregunta debe ser un `str` no vacío y tener como máximo `ckpt['contrato']['max_tokens_pregunta']` tokens; si no, `ValueError`;
2. el modelo: carga `ckpt['contrato']['modelo']`, pon LoRA con `poner_lora(m, ckpt['r'], ckpt['alpha'])` y copia en cada capa sus `A` y `B` de `ckpt['lora']` (las claves son las mismas). Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema (`prompt_ids`), greedy hasta `FIN` y como máximo `ckpt['contrato']['max_nuevos']` tokens.

In [ ]:
_CACHE = {}

def responder_raw(pregunta, ckpt):
    # TODO 4: valida el contrato, arma el modelo con ckpt (una sola vez, en _CACHE) y devuelve la respuesta
    raise NotImplementedError('Completa TODO 4')

In [ ]:
ckpt = torch.load(mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo.pt'), weights_only=False)
for q in test.instruction[:3]:
    a, b = responder_raw(q, ckpt), responder(q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, ckpt); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que $J$ y tu lectura no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y un archivo de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

**Si quieres ir más allá:** escribe un juez simple del estilo (por ejemplo, la fracción de respuestas con al menos dos giros de la lista de `ESTILO`) y compáralo con $J$ en las tres condiciones. MIT usa un LLM como juez; ¿qué problemas tendría aquí?